# 05 — Master Join

## Purpose
Combine all clean data streams into one SA3 × year table that powers the dashboard.
This is the single source of truth for all four chapters.

## Input
- `data/clean/stars_timeline.csv` — quality per facility × snapshot (from notebook 01)
- `data/clean/access_sa3.csv` — users per SA3 × year (from notebook 03)
- `data/clean/supply_sa3.csv` — facilities per SA3 × year (from notebook 02)
- `data/clean/pop_clean.csv` — population 65+ per SA3 × year (pre-existing ABS data)

## Output
- `data/clean/master_sa3.csv` — the main dashboard input file

## Used in
- All four dashboard chapters (map, scatter, ranked bar, trend)

## Key context
- Join level: SA3 × year — both keys must match exactly
- Years available: 2023, 2024, 2025 (all four inputs cover this window)
- Left join from quality_sa3: SA3 regions with star-rated facilities are the anchor
- Engineered metrics drive the four dashboard visuals:
  - `residential_access_rate` = users per 100 elderly residents (V1 map, V3 bar)
  - `care_gap_index` = access_rate / quality_score (higher = more underserved)
  - `places_per_1000_elderly` = licensed beds per 1,000 pop 65+ (supply pressure)
  - `hcp_high_needs` ratio = waitlist pressure proxy (V2 scatter)

In [ ]:
import pandas as pd
import numpy as np

CLEAN = '../../data/clean'
OUT   = f'{CLEAN}/master_sa3.csv'

In [ ]:
# =============================================================================
# STEP 1: Load all clean input files
# =============================================================================

stars  = pd.read_csv(f'{CLEAN}/stars_timeline.csv')
access = pd.read_csv(f'{CLEAN}/access_sa3.csv')
supply = pd.read_csv(f'{CLEAN}/supply_sa3.csv')
pop    = pd.read_csv(f'{CLEAN}/pop_clean.csv')

print('stars:  ', stars.shape,  '| snapshots:', stars['snapshot'].nunique())
print('access: ', access.shape, '| years:', sorted(access['year'].unique()))
print('supply: ', supply.shape, '| years:', sorted(supply['year'].unique()))
print('pop:    ', pop.shape,    '| years:', sorted(pop['year'].unique()))

# Quick check: do SA3 codes look compatible across files?
print('\nSA3 code samples:')
print('  stars:  ', stars['sa3_code'].dropna().head(3).tolist())
print('  access: ', access['sa3_code'].head(3).tolist())
print('  supply: ', supply['sa3_code'].head(3).tolist())
print('  pop:    ', pop['sa3_code'].head(3).tolist())

In [ ]:
# =============================================================================
# STEP 2: Aggregate star ratings to SA3 × year
# =============================================================================
# Stars data is facility × snapshot. We need SA3 × year.
# Mapping: snapshot_date.year → calendar year (so Feb 2025 snapshot → year 2025)
# This aligns with the GEN and service list files which use the 30 June year-end.

stars['snapshot_date'] = pd.to_datetime(stars['snapshot_date'])
stars['year'] = stars['snapshot_date'].dt.year

quality_sa3 = (
    stars.dropna(subset=['sa3_code'])
    .groupby(['sa3_code', 'sa3_name', 'year', 'mmm_code', 'mmm_region', 'state'])
    .agg(
        n_facilities     = ('Service Name', 'nunique'),
        quality_score    = ('quality_score', 'mean'),
        overall_rating   = ('overall_rating', 'mean'),
        residents_exp    = ('residents_exp', 'mean'),
        staffing         = ('staffing', 'mean'),
        compliance       = ('compliance', 'mean'),
        quality_measures = ('quality_measures', 'mean'),
    )
    .reset_index()
)

n_snapshots = stars.dropna(subset=['sa3_code'])['snapshot'].nunique()
print(f'quality_sa3 shape: {quality_sa3.shape}')
print(f'Years: {sorted(quality_sa3["year"].unique())}')
print(f'SA3 regions with quality data: {quality_sa3["sa3_code"].nunique()}')
print(f'{n_snapshots} quarterly snapshots → {quality_sa3["year"].nunique()} annual aggregates')

In [ ]:
# =============================================================================
# STEP 3: Standardise join keys
# =============================================================================
# SA3 codes must be strings with consistent format across all four inputs.
# The stars file uses float SA3 codes from the service list (e.g. 10102.0);
# GEN and supply files may have integer or string codes.
# We strip whitespace and normalise to string — the actual digit format (5-digit)
# will match as long as no zero-padding differences exist.

for df in [quality_sa3, access, supply, pop]:
    if 'sa3_code' in df.columns:
        df['sa3_code'] = df['sa3_code'].astype(str).str.strip().str.split('.').str[0]
    if 'SA3_CODE_2021' in df.columns:
        df['sa3_code'] = df['SA3_CODE_2021'].astype(str).str.strip()
    if 'year' in df.columns:
        df['year'] = df['year'].astype(int)

print('Join key samples after normalisation:')
print('  quality_sa3:', quality_sa3['sa3_code'].head(3).tolist())
print('  access:     ', access['sa3_code'].head(3).tolist())
print('  supply:     ', supply['sa3_code'].head(3).tolist())
print('  pop:        ', pop['sa3_code'].head(3).tolist())

# Check for code overlap
q_codes = set(quality_sa3['sa3_code'])
a_codes = set(access['sa3_code'])
s_codes = set(supply['sa3_code'])
p_codes = set(pop['sa3_code'])
print(f'\nSA3 code counts: quality={len(q_codes)}, access={len(a_codes)}, supply={len(s_codes)}, pop={len(p_codes)}')
print(f'quality ∩ access: {len(q_codes & a_codes)}')
print(f'quality ∩ supply: {len(q_codes & s_codes)}')
print(f'quality ∩ pop:    {len(q_codes & p_codes)}')

In [ ]:
# =============================================================================
# STEP 4: Master join — all four inputs on SA3 × year
# =============================================================================
# Strategy: left join anchored on quality_sa3 (SA3 regions with rated facilities).
# This means SA3 regions with NO residential facility will be absent from the
# main table — which is intentional, since those regions aren't the story here.
#
# Access and supply use outer joins in case a SA3 has GEN data but no star ratings.

master = (
    quality_sa3
    .merge(
        access[['sa3_code', 'year', 'permanent', 'respite', 'total_residential',
                'hcp_level1', 'hcp_level2', 'hcp_level3', 'hcp_level4',
                'total_homecare', 'hcp_high_needs', 'pct_hcp_high']],
        on=['sa3_code', 'year'], how='left'
    )
    .merge(
        supply[['sa3_code', 'year', 'n_facilities', 'n_residential', 'n_homecare',
                'residential_places', 'homecare_places']],
        on=['sa3_code', 'year'], how='left', suffixes=('_quality', '_supply')
    )
    .merge(
        pop[['sa3_code', 'year', 'pop_65_plus', 'total_pop', 'pct_65_plus']],
        on=['sa3_code', 'year'], how='left'
    )
)

print(f'Master shape: {master.shape}')
print('\nNull counts (key metrics):')
key_metrics = ['quality_score', 'total_residential', 'total_homecare', 'pop_65_plus',
               'residential_places']
print(master[key_metrics].isnull().sum().to_string())

In [ ]:
# =============================================================================
# STEP 5: Engineer derived metrics
# =============================================================================

master['residential_access_rate'] = master['total_residential'] / master['pop_65_plus'] * 100
master['home_care_rate']          = master['total_homecare']    / master['pop_65_plus'] * 100
master['total_access_rate']       = (
    master['total_residential'].fillna(0) + master['total_homecare'].fillna(0)
) / master['pop_65_plus'] * 100
master['care_gap_index']          = master['residential_access_rate'] / master['quality_score']
master['places_per_1000_elderly'] = master['residential_places'] / master['pop_65_plus'] * 1000
master['has_residential']         = (master['n_residential'].fillna(0) > 0).astype(int)
master['has_homecare']            = (master['n_homecare'].fillna(0) > 0).astype(int)
master.replace([np.inf, -np.inf], np.nan, inplace=True)

# NOTE: The Feb 2026 star ratings snapshot creates year=2026 rows, but access/supply/pop
# only cover through 2025. Year 2026 rows will have NaN for all access-derived metrics.
# Use year 2025 as the "latest complete" year for current-state analysis.
latest_complete_year = master[master['care_gap_index'].notna()]['year'].max()
print(f'Years in master: {sorted(master["year"].unique())}')
print(f'Latest year with complete data: {latest_complete_year}')

print('\nDerived metrics sample (worst care gap, latest complete year):')
worst = (
    master[master['year'] == latest_complete_year]
    .dropna(subset=['care_gap_index'])
    .sort_values('care_gap_index', ascending=False)
    .head(10)
    [['sa3_name', 'state', 'mmm_code', 'residential_access_rate',
      'quality_score', 'care_gap_index', 'pop_65_plus']]
)
print(worst.round(2).to_string(index=False))

In [ ]:
# =============================================================================
# STEP 6: Save
# =============================================================================

master.to_csv(OUT, index=False)
print(f'Saved: {master.shape[0]:,} rows × {master.shape[1]} columns → {OUT}')
print(f'Years: {sorted(master["year"].unique())}')
print(f'SA3 regions: {master["sa3_code"].nunique()}')
print(f'SA3 with quality data:      {master["quality_score"].notna().sum()}')
print(f'SA3 with access data:       {master["total_residential"].notna().sum()}')
print(f'SA3 with supply data:       {master["residential_places"].notna().sum()}')
print(f'SA3 with population data:   {master["pop_65_plus"].notna().sum()}')
print(f'SA3 with care_gap_index:    {master["care_gap_index"].notna().sum()}')
print('\nAll columns:')
print(master.columns.tolist())